# Chart H5/H20 — 2025 final holdout (Colab)

Drive의 동일 processed 스냅샷으로 두 3분류 LightGBM을 학습한다. H5는 dynamic sigma `u1.75/d1.50`, H20은 `u3.75/d3.00`이다. 2025는 학습·튜닝에 사용하지 않는 최종 OOS 구간이다.

중요: H20 라벨러는 최대 50개 미래 거래행을 탐색한다. 2026년 버퍼가 있으면 2025년 말까지 평가할 수 있고, 부족하면 미래 결과를 확인할 수 없는 마지막 tail만 제외한다. preflight는 profile별 실제 labelable 행이 없는 종목만 제외한다.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 1. 실행 변수
`COMMIT_SHA`는 이 노트북과 공식 holdout config가 포함된 merge commit으로 반드시 고정한다. `DATA_SOURCE`는 `*.tar.gz` 또는 압축 해제된 processed 디렉터리다.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/OWNER/REPOSITORY.git'  # 수정
COMMIT_SHA = 'SET_ME_AFTER_MERGE'                     # 수정: branch 이름 금지
DRIVE_ROOT = Path('/content/drive/MyDrive/stock_prediction/chart_holdout_2025')
DATA_SOURCE = DRIVE_ROOT / 'input/chart_processed_v1.tar.gz'  # 또는 processed 디렉터리
RUN_ROOT = Path('/content/chart_holdout_run')
EXPORT_ROOT = DRIVE_ROOT / 'outputs' / COMMIT_SHA[:12]
RUN_BACKTEST = True
HASH_SOURCE_ARCHIVE = True

if COMMIT_SHA == 'SET_ME_AFTER_MERGE' or len(COMMIT_SHA) < 7:
    raise ValueError('COMMIT_SHA를 이 노트북이 포함된 실제 Git commit으로 고정하세요.')
if 'OWNER/REPOSITORY' in REPO_URL:
    raise ValueError('REPO_URL을 팀 저장소 주소로 수정하세요.')
if not DATA_SOURCE.exists():
    raise FileNotFoundError(DATA_SOURCE)

In [ ]:
import shutil
import subprocess

REPO_DIR = RUN_ROOT / 'repo'
if RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)
RUN_ROOT.mkdir(parents=True)
subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT_SHA], cwd=REPO_DIR, check=True)
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert actual_sha.startswith(COMMIT_SHA), (actual_sha, COMMIT_SHA)
print('Pinned commit:', actual_sha)

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'backend/analysis/chart/requirements.txt')], check=True)

## 2. processed 스냅샷 준비
archive는 `*.tar.gz`/`*.tgz`만 허용한다. 압축을 풀었을 때 종목별 parquet를 재귀 탐색해 단일 실행 디렉터리로 연결한다.

In [ ]:
import tarfile

DATA_DIR = RUN_ROOT / 'processed'
DATA_DIR.mkdir()
if DATA_SOURCE.is_dir():
    source_files = sorted(DATA_SOURCE.rglob('*.parquet'))
else:
    if not (DATA_SOURCE.name.endswith('.tar.gz') or DATA_SOURCE.name.endswith('.tgz')):
        raise ValueError('지원 archive 형식은 .tar.gz/.tgz입니다.')
    extract_dir = RUN_ROOT / 'extracted'
    extract_dir.mkdir()
    with tarfile.open(DATA_SOURCE, 'r:gz') as archive:
        archive.extractall(extract_dir, filter='data')
    source_files = sorted(extract_dir.rglob('*.parquet'))
if not source_files:
    raise RuntimeError('processed parquet를 찾지 못했습니다.')
for source in source_files:
    target = DATA_DIR / source.name
    if target.exists():
        raise RuntimeError(f'중복 파일명: {source.name}')
    target.symlink_to(source.resolve())
print(f'processed files: {len(source_files):,}')

## 3. coverage 및 스키마 preflight
평가 대상 종목마다 2022~2024 학습 라벨과 2025 holdout 라벨이 최소 1개 이상 생성되는지 검사한다. H5는 마지막 13행, H20은 마지막 50행의 미래 관측이 부족하면 해당 tail만 라벨에서 빠진다. 상장폐지 종목은 폐지 전의 유효한 holdout 라벨을 유지하며, 종목 전체를 버리지 않는다. 적격 종목과 제외 이유는 profile별로 CSV에 기록한다.

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

required = {'Date','Code','Open','High','Low','Close','Volume','Trading_Halt','Sigma'}
records = []
for path in sorted(DATA_DIR.glob('*.parquet')):
    names = set(pq.read_schema(path).names)
    missing = sorted(required - names)
    if missing:
        records.append({'Code': path.stem.zfill(6), 'schema_ok': False, 'reason': f'missing:{missing}'})
        continue
    dates = pd.to_datetime(pd.read_parquet(path, columns=['Date'])['Date']).sort_values()
    train_n = int(dates.between('2022-01-01','2024-12-31').sum())
    test_dates = dates[dates.between('2025-01-08','2025-12-31')]
    last_test = test_dates.max() if len(test_dates) else pd.NaT
    future_n = int((dates > last_test).sum()) if pd.notna(last_test) else 0
    records.append({'Code': path.stem.zfill(6), 'schema_ok': True, 'reason': '', 'train_rows': train_n, 'test_rows': len(test_dates), 'last_test_date': last_test, 'future_rows': future_n, 'file_max_date': dates.max()})
coverage = pd.DataFrame(records)
for profile, buffer_rows in {'h5': 13, 'h20': 50}.items():
    coverage[f'labelable_train_rows_{profile}'] = (coverage['train_rows'].fillna(0) - buffer_rows).clip(lower=0)
    missing_right_buffer = (buffer_rows - coverage['future_rows'].fillna(0)).clip(lower=0)
    coverage[f'labelable_test_rows_{profile}'] = (coverage['test_rows'].fillna(0) - missing_right_buffer).clip(lower=0)
    coverage[f'eligible_{profile}'] = coverage['schema_ok'].fillna(False) & coverage[f'labelable_train_rows_{profile}'].gt(0) & coverage[f'labelable_test_rows_{profile}'].gt(0)
display(coverage[['eligible_h5','eligible_h20','future_rows']].describe(include='all'))
if not coverage['eligible_h5'].any() or not coverage['eligible_h20'].any():
    raise RuntimeError('학습 및 2025 holdout 라벨을 생성할 수 있는 universe가 없습니다.')
print('eligible H5:', int(coverage.eligible_h5.sum()), 'eligible H20:', int(coverage.eligible_h20.sum()))

## 4. 실행 config 고정
Git의 공식 template을 복사한 뒤 Drive 스냅샷 절대경로, 스냅샷 식별자, preflight 적격 종목 목록만 주입한다. 원본 template은 수정하지 않는다.

In [ ]:
import hashlib
import json

import yaml


def sha256_file(path, chunk=1024*1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()

snapshot_items = []
for path in sorted(source_files):
    stat = path.stat()
    snapshot_items.append(f'{path.name}:{stat.st_size}:{stat.st_mtime_ns}')
snapshot_id = hashlib.sha256('\n'.join(snapshot_items).encode()).hexdigest()
CONFIG_DIR = RUN_ROOT / 'configs'
CONFIG_DIR.mkdir()
profiles = {
    'h5': ('holdout_2025_h5.yaml', coverage.loc[coverage.eligible_h5, 'Code'].tolist()),
    'h20': ('holdout_2025_h20.yaml', coverage.loc[coverage.eligible_h20, 'Code'].tolist()),
}
for _profile, (filename, tickers) in profiles.items():
    template = REPO_DIR / 'backend/analysis/chart/experiments/configs' / filename
    cfg = yaml.safe_load(template.read_text())
    cfg['data']['price_dir'] = str(DATA_DIR)
    cfg['data']['version'] = snapshot_id
    cfg['data']['tickers'] = tickers
    (CONFIG_DIR / filename).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
coverage.to_csv(CONFIG_DIR / 'coverage_preflight.csv', index=False)
print('snapshot_id:', snapshot_id)

## 5. 학습·공용 평가·백테스트
두 모델은 기존 `train.py`와 공용 ML/백테스트 평가기를 그대로 사용한다. 첫 실행은 오래 걸린다. 런타임이 끊기면 동일 commit·data·config로 재실행하면 캐시를 재사용한다.

In [ ]:
CHART_DIR = REPO_DIR / 'backend/analysis/chart'
# Colab 연결이 끊겨도 동일 commit/data config의 모델·예측 캐시를 재사용한다.
DRIVE_CACHE = DRIVE_ROOT / 'work_cache' / actual_sha[:12]
for relative in ('experiments/cache', 'experiments/train_src/cache'):
    local_cache = CHART_DIR / relative
    persistent_cache = DRIVE_CACHE / relative
    persistent_cache.mkdir(parents=True, exist_ok=True)
    if local_cache.exists() or local_cache.is_symlink():
        if local_cache.is_symlink():
            local_cache.unlink()
        else:
            shutil.rmtree(local_cache)
    local_cache.parent.mkdir(parents=True, exist_ok=True)
    local_cache.symlink_to(persistent_cache, target_is_directory=True)
for profile, (filename, _) in profiles.items():
    config_path = CONFIG_DIR / filename
    print(f'===== TRAIN {profile.upper()} =====')
    subprocess.run([sys.executable, 'experiments/train.py', '--config', str(config_path)], cwd=CHART_DIR, check=True)
    print(f'===== ML EVALUATION {profile.upper()} =====')
    subprocess.run([sys.executable, 'experiments/run_ml_evaluation.py', '--config', str(config_path)], cwd=CHART_DIR, check=True)
    if RUN_BACKTEST:
        print(f'===== BACKTEST {profile.upper()} =====')
        subprocess.run([sys.executable, 'experiments/run_backtest.py', '--config', str(config_path)], cwd=CHART_DIR, check=True)

## 6. 실제 모델·결과를 Drive로 export
예측 hash로 실제 cache artifact를 찾아 canonical 이름으로 복사한다. 가짜/빈 모델은 만들지 않으며 산출물이 없으면 실패한다.

In [ ]:
import platform

sys.path.insert(0, str(CHART_DIR / 'experiments'))
import lightgbm
import numpy
import sklearn
from experiment_utils import generate_predictions_hash, resolve_splits

if EXPORT_ROOT.exists():
    raise FileExistsError(f'기존 export를 덮어쓰지 않습니다: {EXPORT_ROOT}')
(EXPORT_ROOT / 'models').mkdir(parents=True)
(EXPORT_ROOT / 'predictions').mkdir()
(EXPORT_ROOT / 'results').mkdir()
(EXPORT_ROOT / 'configs').mkdir()
canonical = {
 'h5': 'baseline_h5_u175_d150_train2022_2024_holdout2025.txt',
 'h20': 'baseline_h20_u375_d300_train2022_2024_holdout2025.txt',
}
artifact_records = []
for profile, (filename, tickers) in profiles.items():
    config_path = CONFIG_DIR / filename
    cfg = yaml.safe_load(config_path.read_text())
    pred_hash = generate_predictions_hash(cfg, resolve_splits(cfg))
    model_source = CHART_DIR / 'experiments/train_src/cache/models' / f'{pred_hash}_fold0_model.txt'
    prediction_source = CHART_DIR / 'experiments/cache' / f'{pred_hash}_predictions.parquet'
    result_source = CHART_DIR / 'experiments/results' / cfg['experiment_name']
    for required_path in (model_source, prediction_source, result_source):
        if not required_path.exists():
            raise FileNotFoundError(required_path)
    model_target = EXPORT_ROOT / 'models' / canonical[profile]
    pred_target = EXPORT_ROOT / 'predictions' / f'{profile}_2025_predictions.parquet'
    shutil.copy2(model_source, model_target)
    shutil.copy2(prediction_source, pred_target)
    shutil.copytree(result_source, EXPORT_ROOT / 'results' / profile)
    shutil.copy2(config_path, EXPORT_ROOT / 'configs' / filename)
    artifact_records.append({'profile': profile, 'prediction_hash': pred_hash, 'eligible_tickers': len(tickers), 'model_file': str(model_target.relative_to(EXPORT_ROOT)), 'model_sha256': sha256_file(model_target), 'predictions_file': str(pred_target.relative_to(EXPORT_ROOT)), 'predictions_sha256': sha256_file(pred_target)})
coverage.to_csv(EXPORT_ROOT / 'coverage_preflight.csv', index=False)
manifest = {
 'schema_version': 1, 'git_commit': actual_sha, 'data_snapshot_id': snapshot_id,
 'data_source': str(DATA_SOURCE),
 'data_source_sha256': sha256_file(DATA_SOURCE) if DATA_SOURCE.is_file() and HASH_SOURCE_ARCHIVE else None,
 'periods': {'train': ['2022-01-01','2024-12-31'], 'holdout': ['2025-01-08','2025-12-31']},
 'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': numpy.__version__,
 'lightgbm': lightgbm.__version__, 'scikit_learn': sklearn.__version__,
 'artifacts': artifact_records,
}
manifest_path = EXPORT_ROOT / 'artifact_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
hash_lines = []
for path in sorted(p for p in EXPORT_ROOT.rglob('*') if p.is_file() and p.name != 'SHA256SUMS'):
    hash_lines.append(f'{sha256_file(path)}  {path.relative_to(EXPORT_ROOT)}')
(EXPORT_ROOT / 'SHA256SUMS').write_text('\n'.join(hash_lines) + '\n')
print('Export complete:', EXPORT_ROOT)
print((EXPORT_ROOT / 'SHA256SUMS').read_text())

## 7. 전달 전 확인
Drive의 `artifact_manifest.json`, `SHA256SUMS`, `coverage_preflight.csv`, 두 config, prediction, 공용 평가 결과를 함께 전달한다. 두 canonical model은 평가 결과가 유효한지 검토한 뒤에만 `core/models/`로 반입하고 `registry.yaml`의 SHA-256과 실행 commit을 갱신한다.